In [ ]:
"""
Минимальный BPE tokenizer — показывает зачем нужны байты.

Ключевая идея: любой текст (любой язык, эмодзи, бинарные данные)
можно представить как последовательность байтов (0-255).
Это даёт нам стартовый словарь размером всего 256 токенов,
из которого BPE строит более крупные токены путём слияний.
"""

text = "ааабабааба"  # кириллица — 2 байта на символ в UTF-8

# ===== ШАГ 0: Текст → байты =====
# Вот ЗАЧЕМ байты: "а" в UTF-8 = [0xd0, 0xb0] — два байта.
# Без байтов нам нужен отдельный токен для каждого Unicode-символа
# (сотни тысяч). С байтами стартовый словарь = всего 256 записей.
tokens = list(text.encode("utf-8"))
print(f"Текст:  {text!r}")
print(f"Байты:  {tokens}")
print(f"Длина:  {len(tokens)} токенов (стартовый словарь: 0..255)\n")

# ===== ШАГ 1: Обучение BPE — жадные слияния пар =====
merges = []  # список выученных слияний
num_merges = 8

for i in range(num_merges):
    # Считаем частоту каждой соседней пары
    pairs = {}
    for a, b in zip(tokens, tokens[1:]):
        pairs[(a, b)] = pairs.get((a, b), 0) + 1

    if not pairs:
        break

    # Берём самую частую пару
    best = max(pairs, key=pairs.get)
    new_id = 256 + i  # новый токен
    merges.append((best, new_id))

    # Заменяем все вхождения пары на новый токен
    new_tokens = []
    j = 0
    while j < len(tokens):
        if j < len(tokens) - 1 and (tokens[j], tokens[j + 1]) == best:
            new_tokens.append(new_id)
            j += 2
        else:
            new_tokens.append(tokens[j])
            j += 1
    tokens = new_tokens

    print(f"Слияние {i}: {best} → {new_id}  "
          f"(частота {pairs[best]})  токены: {tokens}")

print(f"\nИтог: {len(text.encode('utf-8'))} байт → {len(tokens)} токенов")
print(f"\n--- Словарь слияний ---")
for (a, b), new_id in merges:
    print(f"  {new_id}: ({a}, {b})")


Текст:  'ааабабааба'
Байты:  [208, 176, 208, 176, 208, 176, 208, 177, 208, 176, 208, 177, 208, 176, 208, 176, 208, 177, 208, 176]
Длина:  20 токенов (стартовый словарь: 0..255)

Слияние 0: (208, 176) → 256  (частота 7)  токены: [256, 256, 256, 208, 177, 256, 208, 177, 256, 256, 208, 177, 256]
Слияние 1: (256, 256) → 257  (частота 3)  токены: [257, 256, 208, 177, 256, 208, 177, 257, 208, 177, 256]
Слияние 2: (208, 177) → 258  (частота 3)  токены: [257, 256, 258, 256, 258, 257, 258, 256]
Слияние 3: (256, 258) → 259  (частота 2)  токены: [257, 259, 259, 257, 258, 256]
Слияние 4: (257, 259) → 260  (частота 1)  токены: [260, 259, 257, 258, 256]
Слияние 5: (260, 259) → 261  (частота 1)  токены: [261, 257, 258, 256]
Слияние 6: (261, 257) → 262  (частота 1)  токены: [262, 258, 256]
Слияние 7: (262, 258) → 263  (частота 1)  токены: [263, 256]

Итог: 20 байт → 2 токенов

--- Словарь слияний ---
  256: (208, 176)
  257: (256, 256)
  258: (208, 177)
  259: (256, 258)
  260: (257, 259)
  261: (260,

In [13]:
text = "ааабабааба"  # кириллица — 2 байта на символ в UTF-8

# ===== ШАГ 0: Текст → байты =====
# Вот ЗАЧЕМ байты: "а" в UTF-8 = [0xd0, 0xb0] — два байта.
# Без байтов нам нужен отдельный токен для каждого Unicode-символа
# (сотни тысяч). С байтами стартовый словарь = всего 256 записей.
print(text.encode("utf-8"))
tokens = list(text.encode("utf-8"))
print(f'ДЛИНА ТЕКСТА: {len(text)}, ЧИСЛО БАЙТОВ {len(tokens)}')
print(f"Текст:  {text!r}")
print(f"Байты:  {tokens}")
print(f"Длина:  {len(tokens)} токенов (стартовый словарь: 0..255)\n")


b'\xd0\xb0\xd0\xb0\xd0\xb0\xd0\xb1\xd0\xb0\xd0\xb1\xd0\xb0\xd0\xb0\xd0\xb1\xd0\xb0'
ДЛИНА ТЕКСТА: 10, ЧИСЛО БАЙТОВ 20
Текст:  'ааабабааба'
Байты:  [208, 176, 208, 176, 208, 176, 208, 177, 208, 176, 208, 177, 208, 176, 208, 176, 208, 177, 208, 176]
Длина:  20 токенов (стартовый словарь: 0..255)



In [2]:
text = "Ababab12"  # кириллица — 2 байта на символ в UTF-8

# ===== ШАГ 0: Текст → байты =====
# Вот ЗАЧЕМ байты: "а" в UTF-8 = [0xd0, 0xb0] — два байта.
# Без байтов нам нужен отдельный токен для каждого Unicode-символа
# (сотни тысяч). С байтами стартовый словарь = всего 256 записей.
print(text.encode("utf-8"))
tokens = list(text.encode("utf-8"))
print(f'ДЛИНА ТЕКСТА: {len(text)}, ЧИСЛО БАЙТОВ {len(tokens)}')
print(f"Текст:  {text!r}")
print(f"Байты:  {tokens}")
print(f"Длина:  {len(tokens)} токенов (стартовый словарь: 0..255)\n")

b'Ababab12'
ДЛИНА ТЕКСТА: 8, ЧИСЛО БАЙТОВ 8
Текст:  'Ababab12'
Байты:  [65, 98, 97, 98, 97, 98, 49, 50]
Длина:  8 токенов (стартовый словарь: 0..255)



In [14]:
merges = []  # список выученных слияний
num_merges = 8
text = "ааабабааба"  # кириллица — 2 байта на символ в UTF-8

print(f'INPUT TEXT: {text}')
for i in range(num_merges):
    # Считаем частоту каждой соседней пары
    pairs = {}
    for a, b in zip(tokens, tokens[1:]):
        pairs[(a, b)] = pairs.get((a, b), 0) + 1

    if not pairs:
        break

    # Берём самую частую пару
    best = max(pairs, key=pairs.get)
    new_id = 256 + i  # новый токен
    merges.append((best, new_id))

    # Заменяем все вхождения пары на новый токен
    new_tokens = []
    j = 0
    while j < len(tokens):
        if j < len(tokens) - 1 and (tokens[j], tokens[j + 1]) == best:
            new_tokens.append(new_id)
            j += 2
        else:
            new_tokens.append(tokens[j])
            j += 1
    tokens = new_tokens

    print(f"Слияние {i}, {chr(best[0])} {chr(best[1])}: {best} → {new_id}  "
          f"(частота {pairs[best]})  токены: {tokens}")

print(f"\nИтог: {len(text.encode('utf-8'))} байт → {len(tokens)} токенов")
print(f"\n--- Словарь слияний ---")
for (a, b), new_id in merges:
    print(f"  {new_id}: ({a}, {b})")

INPUT TEXT: ааабабааба
Слияние 0, Ð °: (208, 176) → 256  (частота 7)  токены: [256, 256, 256, 208, 177, 256, 208, 177, 256, 256, 208, 177, 256]
Слияние 1, Ā Ā: (256, 256) → 257  (частота 3)  токены: [257, 256, 208, 177, 256, 208, 177, 257, 208, 177, 256]
Слияние 2, Ð ±: (208, 177) → 258  (частота 3)  токены: [257, 256, 258, 256, 258, 257, 258, 256]
Слияние 3, Ā Ă: (256, 258) → 259  (частота 2)  токены: [257, 259, 259, 257, 258, 256]
Слияние 4, ā ă: (257, 259) → 260  (частота 1)  токены: [260, 259, 257, 258, 256]
Слияние 5, Ą ă: (260, 259) → 261  (частота 1)  токены: [261, 257, 258, 256]
Слияние 6, ą ā: (261, 257) → 262  (частота 1)  токены: [262, 258, 256]
Слияние 7, Ć Ă: (262, 258) → 263  (частота 1)  токены: [263, 256]

Итог: 20 байт → 2 токенов

--- Словарь слияний ---
  256: (208, 176)
  257: (256, 256)
  258: (208, 177)
  259: (256, 258)
  260: (257, 259)
  261: (260, 259)
  262: (261, 257)
  263: (262, 258)


In [19]:
chr(208), chr(176)

('Ð', '°')

In [ ]:
a - 256
aa - 257
b - 258
ab  - 259

6

In [20]:
len([256, 256, 256, 208, 177, 256, 208, 177, 256, 256, 208, 177, 256])

13

In [ ]:
[208, 176, 208, 176, 208, 176, 208, 177, 208, 176, 208, 177, 208, 176, 208, 176, 208, 177, 208, 176]

256

In [16]:
chr(208), chr(176)

('Ð', '°')